## Import useful modules ⬇️⬇️


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## File reading and basic exploration 📰📰


In [4]:
# Import dataset
print("Loading dataset...")
workdir = "/home/app/M04-Machine_Learning/D01-Preprocessing_with_Sklearn/00-Lectures/src" # Change the path of the working directory your file is in
dataset = pd.read_csv("Dataset_preprocessing.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [5]:
dataset.describe()

,id,Age,Salary,almost_empty
count,12.000000,11.000000,1.200000e+01,2.000000
mean,5.500000,36.909091,8.338958e+07,30.000000
std,3.605551,19.002392,2.886574e+08,14.142136
min,0.000000,-10.000000,3.200000e+04,20.000000
25%,2.750000,32.500000,5.350000e+04,25.000000
50%,5.500000,38.000000,6.400000e+04,30.000000
75%,8.250000,46.000000,7.375000e+04,35.000000
max,11.000000,67.000000,1.000000e+09,40.000000


In [10]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include="all")
display(data_desc)
print()

print("Percentage of missing values: ")
display(100 * dataset.isnull().sum() / dataset.shape[0])

Number of rows : 12

Display of dataset: 


,id,Country,Age,Salary,Purchased,useless_col,almost_empty
0,0,France,44.0,72000,No,useless,NaN
1,1,Spain,27.0,48000,Yes,useless,40.0
2,2,Germany,30.0,54000,No,useless,NaN
3,3,Spain,38.0,61000,No,useless,20.0
4,4,Germany,40.0,69000,Yes,useless,NaN



Basics statistics: 


,id,Country,Age,Salary,Purchased,useless_col,almost_empty
count,12.000000,12,11.000000,1.200000e+01,12,12,2.000000
unique,NaN,3,NaN,NaN,2,1,NaN
top,NaN,France,NaN,NaN,Yes,useless,NaN
freq,NaN,5,NaN,NaN,7,12,NaN
mean,5.500000,NaN,36.909091,8.338958e+07,NaN,NaN,30.000000
std,3.605551,NaN,19.002392,2.886574e+08,NaN,NaN,14.142136
min,0.000000,NaN,-10.000000,3.200000e+04,NaN,NaN,20.000000
25%,2.750000,NaN,32.500000,5.350000e+04,NaN,NaN,25.000000
50%,5.500000,NaN,38.000000,6.400000e+04,NaN,NaN,30.000000
75%,8.250000,NaN,46.000000,7.375000e+04,NaN,NaN,35.000000



Percentage of missing values: 


id               0.000000
Country          0.000000
Age              8.333333
Salary           0.000000
Purchased        0.000000
useless_col      0.000000
almost_empty    83.333333
dtype: float64

L’exploration des données ci-dessus permet de savoir quelles étapes de prétraitement seront nécessaires :

### I- Prétraitement à planifier avec pandas

**Colonnes inutiles pour la prédiction, à supprimer** :

* `id` est un identifiant, il ne doit jamais être utilisé pour la prédiction (cette colonne ne contient aucune information)
* `useless_col` sera également inutile, car elle contient toujours la même valeur

**Colonnes avec trop de valeurs manquantes, à écarter** :

* `almost_empty`

**Lignes contenant des valeurs aberrantes, à supprimer** :

* Lignes pour lesquelles `Age` est négatif
* Lignes pour lesquelles `Salary` est à plus de 2 écarts-types (`std`) de la moyenne

---

> Note :  
> Concernant le Salary, la suppression des lignes situées à plus de 2 écarts-types de la moyenne provient de la distribution normale (gaussienne) selon laquelle :
> * ~68 % de vos données totales se trouvent à 1 écart-type de la moyenne
> * ~95 % de vos données totales se trouvent à 2 écarts-types de la moyenne
> * ~99 % de vos données totales se trouvent à 3 écarts-types de la moyenne
> En supposant que la distribution des salaires suit une distribution gaussienne, nous pouvons considérer comme valeurs aberrantes toutes les données situées à plus de 2 écarts-types de la moyenne des salaires.  

---


**Variable cible (Y) que nous allons essayer de prédire, à séparer des autres** :

* `Purchased`

### II- Prétraitements à planifier avec scikit-learn

**Variables explicatives (X)**

Il est nécessaire d’identifier quelles colonnes contiennent des variables catégorielles et quelles colonnes contiennent des variables numériques, car elles seront traitées différemment.

* Variables catégorielles : `Country`
* Variables numériques : `Age`, `Salary`

Dans ce jeu de données, nous avons les deux types de variables. Il sera donc nécessaire de prévoir la création d’un `numeric_transformer` (qui appellera la classe `StandardScaler`) et d’un `categorical_transformer` (qui appellera la classe `OneHotEncoder`). D’autre part, comme nous observons des valeurs manquantes dans le jeu de données de départ, il faudra prévoir l’utilisation de la classe `SimpleImputer` pour gérer ces valeurs manquantes.

**Variable cible Y**
Enfin, il convient de noter que la variable Y à prédire est catégorielle : il sera donc nécessaire de prévoir un encodage avec la classe `LabelEncoder`.


## Preprocessing - Pandas 🐼🐼


In [11]:
# Drop useless columns / columns with too many missing values
useless_cols = ["id", "useless_col", "almost_empty"]

print("Dropping useless columns...")
dataset = dataset.drop(
    useless_cols, axis=1
)  # axis = 1 indicates that we are dropping along the column axis
# never hesitate to look at a function's documentation using the command name_of_the_function?
print("...Done.")
display(dataset.head())

Dropping useless columns...
...Done.


,Country,Age,Salary,Purchased
0,France,44.0,72000,No
1,Spain,27.0,48000,Yes
2,Germany,30.0,54000,No
3,Spain,38.0,61000,No
4,Germany,40.0,69000,Yes


In [12]:
# Drop lines containing outliers (using masks)

print("Dropping outliers in Age...")
to_keep = (dataset["Age"] > 0) | (
    dataset["Age"].isnull()
)  # We want keeping positives values or missings
dataset = dataset.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()

print("Dropping outliers in Salary...")
to_keep = dataset["Salary"] < dataset["Salary"].mean() + 2 * dataset["Salary"].std()
dataset = dataset.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()

dataset.head()

Dropping outliers in Age...
Done. Number of lines remaining :  11

Dropping outliers in Salary...
Done. Number of lines remaining :  10



,Country,Age,Salary,Purchased
0,France,44.0,72000,No
1,Spain,27.0,48000,Yes
2,Germany,30.0,54000,No
3,Spain,38.0,61000,No
4,Germany,40.0,69000,Yes


In [13]:
# Separate target variable Y from features X
target_name = "Purchased"

print("Separating labels from features...")
Y = dataset.loc[:, target_name]
X = dataset.drop(target_name, axis=1)  # All columns are kept, except the target
print("...Done.")
display(Y.head())
print()
display(X.head())
print()

Separating labels from features...
...Done.


0     No
1    Yes
2     No
3     No
4    Yes
Name: Purchased, dtype: str

,Country,Age,Salary
0,France,44.0,72000
1,Spain,27.0,48000
2,Germany,30.0,54000
3,Spain,38.0,61000
4,Germany,40.0,69000


## Prétraitement - Scikit-Learn 🔬🔬

### Séparation train-test

Avant de réaliser tout prétraitement, mettons de côté une petite partie (10-20 %) du jeu de données. Ces données seront appelées le *jeu de test* et ne seront pas utilisées pour entraîner le modèle.  
Plus tard dans ce notebook, nous utiliserons la plus grande partie (80-90 %) des données (le *jeu d’entraînement*) pour entraîner notre modèle, puis le jeu de test nous permettra d’estimer les performances que nous pourrions attendre sur de nouvelles données jamais vues par le modèle.

---

>**TIP**:  
>Si l’importance de cette séparation train/test n’est pas encore évidente pour vous, ne vous inquiétez pas : nous approfondirons ce sujet à la fin du module 🤓. Pour l’instant, vous pouvez simplement appliquer cette règle simple : faites toujours une séparation train/test avant d’appliquer toute transformation sur vos données avec scikit-learn !



In [14]:
# First : always divide dataset into train set & test set !!
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
# test_size indicates the proportion of rows from X and Y that will go into the test dataset while
# maintaining the correspondance between the rows from X and Y

# random_state is an argument that can be found in all functions that have a pseudo-random behaviour
# if random_state is not stated the function will derive a different random result everytime the cell
# runs, if random_state is given a value the results will be the same everytime the cell runs while
# each different value of radom_state will derive a specific result
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Prétraitements : imputation des valeurs manquantes, standardisation et encodage one-hot

Appliquons maintenant les transformations restantes nécessaires pour préparer les données à l’entraînement du modèle.

Il existe une difficulté principale : nous voulons appliquer des transformations différentes aux variables, selon leur type. Les colonnes numériques seront standardisées tandis que les colonnes catégorielles seront encodées en one-hot. Quant à l’imputation des valeurs manquantes, elle peut s’appliquer à toutes les colonnes, mais des règles différentes peuvent être utilisées. Par exemple, on pourra remplacer les valeurs manquantes par la médiane pour les variables numériques, et par le mode de la variable pour les variables catégorielles. On peut imaginer qu’avec de nombreuses colonnes de types différents, cela peut conduire à beaucoup de code avec de nombreuses exceptions 🤯

Heureusement, les classes `Pipeline` et `ColumnTransformer` de scikit-learn vont nous permettre d’écrire un code propre et lisible tout en appliquant différentes étapes selon les colonnes.

>Note : Pipeline vs. ColumnTransformer
>
>* La classe `Pipeline` permet de décrire les étapes successives que l’on souhaite appliquer à une variable. Par exemple : « 1. remplacer les valeurs manquantes par la médiane 2. standardiser la variable ». Une fois ce pipeline déclaré, il est possible de l’appliquer à chaque colonne concernée.
>* La classe `ColumnTransformer` permet d’appliquer un `Pipeline` donné à un ensemble de colonnes d’un coup.

---

> Warning -> Pas de panique !
>
>Cela peut sembler un peu compliqué au début, mais vous utiliserez ces classes tous les jours à partir de maintenant, et vous verrez que le comportement est toujours le même. Au final, il est probable que vous connaissiez ces lignes de code par cœur ; répéter et pratiquer est la clé 😌. Mais au début, vous pouvez simplement vous contenter de copier/coller ces lignes dès que vous devez effectuer un prétraitement !



In [15]:
# Create pipeline for numeric features
numeric_features = ["Age", "Salary"]  # Names of numeric columns in X_train/X_test
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),  # missing values will be replaced by columns' median
        ("scaler", StandardScaler()),
    ]
)

In [16]:
# Create pipeline for categorical features
categorical_features = ["Country"]  # Names of categorical columns in X_train/X_test
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),  # missing values will be replaced by most frequent value
        (
            "encoder",
            OneHotEncoder(drop="first"),
        ),  # first column will be dropped to avoid creating correlations between features
    ]
)

In [17]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [18]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print("...Done.")
print(
    X_train[0:5]
)  # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(
    X_test
)  # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print("...Done.")
print(
    X_test[0:5, :]
)  # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
   Country   Age  Salary
4  Germany  40.0   69000
9   France  37.0   67000
1    Spain  27.0   48000
6    Spain   NaN   52000
7   France  48.0   79000
...Done.
[[ 0.27978024  0.58858382  1.          0.        ]
 [-0.23673712  0.38385901  0.          0.        ]
 [-1.95846165 -1.56102665  0.          1.        ]
 [-0.06456467 -1.15157703  0.          1.        ]
 [ 1.65715986  1.61220785  0.          0.        ]]

Performing preprocessings on test set...
   Country   Age  Salary
2  Germany  30.0   54000
8  Germany  50.0   83000
...Done.
[[-1.44194429 -0.94685223  1.          0.        ]
 [ 2.00150476  2.02165746  1.          0.        ]]



>Warning -> Conversion de type
>
>Le `ColumnTransformer` de scikit-learn convertit les DataFrames pandas en tableaux numpy. Vous pouvez vérifier cela en affichant le type de l’objet retourné par la méthode `fit_transform()`. La cellule ci-dessous ne s’exécutera pas correctement, car toutes les méthodes propres à pandas ne seront plus disponibles une fois la conversion effectuée.  
>Pour afficher les premières lignes de votre jeu de données, vous devrez utiliser la syntaxe numpy comme montré ci-dessous.



In [19]:
# This will create an AttributeError because X_train is not a pandas DataFrame anymore!
print("IT IS EXPECTED TO GET AN ERROR HERE 😉")
X_train.head()

IT IS EXPECTED TO GET AN ERROR HERE 😉


AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [ ]:
# Actually, X_train is a numpy array
print("THIS IS THE ACTUAL TYPE OF DATA YOU ARE DEALING WITH NOW 👇")
type(X_train)

THIS IS THE ACTUAL TYPE OF DATA YOU ARE DEALING WITH NOW 👇


numpy.ndarray

In [ ]:
# Numpy syntax to display 5 first lines
X_train[0:5]

array([[ 0.27978024,  0.58858382,  1.        ,  0.        ],
       [-0.23673712,  0.38385901,  0.        ,  0.        ],
       [-1.95846165, -1.56102665,  0.        ,  1.        ],
       [-0.06456467, -1.15157703,  0.        ,  1.        ],
       [ 1.65715986,  1.61220785,  0.        ,  0.        ]])

In [ ]:
# Encode target variable Y
labelencoder = LabelEncoder()

print("Encoding labels on train set...")
print(Y_train.head())
print()
Y_train = labelencoder.fit_transform(Y_train)
print("...Done.")
print(Y_train[0:5])
print()

print("Encoding labels on test set...")
print(Y_test.head())
print()
Y_test = labelencoder.transform(Y_test)  # Don't fit again !!
print("...Done.")
print(Y_test[0:5])
print()

Encoding labels on train set...
4    Yes
9    Yes
1    Yes
6     No
7    Yes
Name: Purchased, dtype: object

...Done.
[1 1 1 0 1]

Encoding labels on test set...
2    No
8    No
Name: Purchased, dtype: object

...Done.
[0 0]



### Model training 🏃


In [ ]:
# Train model
model = LogisticRegression()

print("Training model...")
model.fit(X_train, Y_train)  # Training is always done on train set !!
print("...Done.")

Training model...


NameError: name 'X_train' is not defined

### Predictions 🔮


In [ ]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = model.predict(X_train)
print("...Done.")
print(Y_train_pred[0:5])
print()

Predictions on training set...
...Done.
[1 1 1 0 1]



In [ ]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = model.predict(X_test)
print("...Done.")
print(Y_test_pred[0:5])
print()

Predictions on test set...
...Done.
[1 1]



### Évaluation des performances 💯

Pour évaluer les performances de votre modèle, vous pouvez importer la fonction correspondant à la métrique qui vous intéresse depuis le module [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics).  
Il existe de nombreuses méthodes de scoring, vous vous familiariserez avec certaines d’entre elles lors des prochaines séances. Dans cette démonstration, nous utilisons le **score de précision (accuracy)**, qui est la métrique la plus répandue pour les problèmes de classification.

**Ici, la précision sur le jeu de test est faible car il n’y a pas assez de données.**


In [ ]:
# Print scores
print(
    "Accuracy on training set : ", accuracy_score(Y_train, Y_train_pred)
)  # Always pass true label first, and predictions in second position
print("Accuracy on test set : ", accuracy_score(Y_test, Y_test_pred))

Accuracy on training set :  0.75
Accuracy on test set :  0.0


### Autre façon de calculer le score de précision

Dans scikit-learn, chaque classe implémentant un modèle de machine learning possède une méthode `score()` qui permet d’évaluer le score **sans avoir à calculer explicitement les prédictions du modèle au préalable**.

> Warning" -> Vérifiez toujours la métrique utilisée !
>
>La méthode `score()` renvoie un score de base cohérent avec le type de problème (dans cette démonstration, il s’agit de la précision). Pour être certain de la métrique utilisée, vous pouvez vous référer à la [documentation du modèle](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.score).


In [ ]:
# Print scores
print(
    "Accuracy on training set : ", model.score(X_train, Y_train)
)  # Here, the features must be passed first, and then the true label
print("Accuracy on test set : ", model.score(X_test, Y_test))

Accuracy on training set :  0.75
Accuracy on test set :  0.0


## Resources 📚📚

- Introduction Pratique à Python - Antoine Krajnc-Rosenthal & Anais Armandy
- Sklearn - [Documentation officielle de scikit-learn](https://scikit-learn.org/stable/)
